# পাঠ ০১ - এআই এজেন্টের পরিচয়

**নবীনদের জন্য AI এজেন্ট** কোর্সের প্রথম পাঠে আপনাকে স্বাগতম!

একটি **এআই এজেন্ট** হল একটি প্রোগ্রাম যা একটি বড় ভাষার মডেল (LLM) কে তার যৌক্তিক ইঞ্জিন হিসেবে ব্যবহার করে এবং বাস্তব জগতে *কর্ম* নিতে পারে — যেমন API কল করা, ডাটাবেস প্রশ্ন করা, বা কোড চালানো — একটি লক্ষ্য অর্জনের জন্য ব্যবহারকারীর পক্ষে।

এই নোটবুকে আপনি আপনার প্রথম এজেন্ট তৈরি করবেন: একটি **ট্র্যাভেল এজেন্ট** যা ছুটির গন্তব্যস্থান সাজেস্ট করে। পথিমধ্যে আপনি শিখবেন কিভাবে:

১. **মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক** ব্যবহার করে Azure AI Foundry Agent Service এর সাথে সংযোগ স্থাপন করবেন।

২. এজেন্টকে একটি **টুল** দেবেন — একটি সাধারণ পাইথন ফাংশন যা এজেন্ট কল করতে পারে।

৩. এজেন্ট চালাবেন এবং তার প্রতিক্রিয়া পরিদর্শন করবেন।

৪. এজেন্টের প্রতিক্রিয়া টোকেন-বাই-টোকেন স্ট্রিম করবেন।


## সেটআপ

এই নোটবুক চালানোর আগে নিশ্চিত করুন আপনার কাছে:

1. **একটি Azure AI Foundry প্রোজেক্ট** যার সাথে একটি ডিপ্লয় করা চ্যাট মডেল আছে (যেমন `gpt-4o-mini`)।
2. **Azure CLI দিয়ে লগ ইন করেছেন** — আপনার টার্মিনালে `az login` চালান।
3. **প্রয়োজনীয় পরিবেশ পরিবর্তনশীল সেট করেছেন:**
   - `AZURE_AI_PROJECT_ENDPOINT` — আপনার Azure AI Foundry প্রোজেক্টের এন্ডপয়েন্ট।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — আপনার ডিপ্লয় করা মডেলের নাম।

নিম্নে থাকা সেলটি আপনার প্রয়োজনীয় পাইথন প্যাকেজগুলি ইনস্টল করে।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## আপনার প্রথম এজেন্ট তৈরি করা

একটি এজেন্টের দুইটি জিনিস প্রয়োজন:

- **নির্দেশনা** যা তাকে বলে *সে কে* এবং *কিভাবে আচরণ করবে* (একটি সিস্টেম প্রম্পট)।
- **টুলস** — পাইথন ফাংশন যা `@tool` দিয়ে সজ্জিত থাকে এবং যে টুলগুলো এজেন্ট তথ্য পুনরুদ্ধার বা কাজ সম্পাদনের জন্য কল করতে পারে।

নীচে আমরা একটি সহজ টুল সংজ্ঞায়িত করেছি যা জনপ্রিয় ছুটির গন্তব্যের একটি তালিকা ফিরিয়ে দেয়। ব্যবহারকারী যখন ভ্রমণের সুপারিশ চায় তখন এজেন্ট এই টুলটি ব্যবহার করবে।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## স্ট্রিমিং রেসপন্সেস

একটি আরও ইন্টারেক্টিভ অভিজ্ঞতার জন্য আপনি এজেন্টের উত্তর **স্ট্রিম** করতে পারেন। সম্পূর্ণ উত্তরটির জন্য অপেক্ষা করার পরিবর্তে, এজেন্ট তৈরি হওয়া টেক্সটের অংশগুলো সরবরাহ করে। এটি বিশেষ করে চ্যাট ইন্টারফেসে উপযোগী যেখানে আপনি রিয়েল টাইমে আউটপুট প্রদর্শন করতে চান।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## সারাংশ

এই পাঠে আপনি শিখেছেন কীভাবে:

- **একটি প্রোভাইডার তৈরি করবেন** যা `AzureAIProjectAgentProvider` এর মাধ্যমে Azure AI Foundry Agent Service-এর সাথে কানেক্ট করে।
- **`@tool` ডেকোরেটর ব্যবহার করে একটি টুল নির্ধারণ করবেন** যাতে এজেন্ট আপনার পাইথন ফাংশনগুলো কল করতে পারে।
- **ব্যবহারকারীর মেসেজ দিয়ে এজেন্ট চালাবেন** এবং এর প্রতিক্রিয়া প্রিন্ট করবেন।
- **রিয়েল-টাইম আউটপুটের জন্য প্রতিক্রিয়াগুলো স্ট্রিম করবেন**।

পরবর্তী পাঠে আমরা এজেন্টিক ফ্রেমওয়ার্কগুলিকে আরও গভীরভাবে অন্বেষণ করব এবং এজেন্টদের আরো শক্তিশালী টুল এবং মাল্টি-স্টেপ রিজনিং ক্ষমতা দেওয়ার পদ্ধতি শিখব।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসংগতি থাকতে পারে। মূল নথিটি তার নিজস্ব ভাষায় সর্বোত্তম এবং প্রামাণিক উৎস হিসেবে গ্রহণ করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানুষের দ্বারা অনুবাদ করার পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারের ফলে হওয়া কোন ধরনের ভুল বোঝাবুঝি বা অস্পষ্টতার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
